# 02 · Features, preparación y validación
Las features derivadas viven en `src/pipeline.py`; este notebook las importa y presenta la evidencia.
Ejecutar con *Restart* y luego *Run All*.

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from src.datos import cargar_obt
from src.pipeline import *
sns.set_theme(style="whitegrid"); FIG="/work/docs/figs/"; os.makedirs(FIG, exist_ok=True)
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)

## 1. División única train / hold-out (temporal)
El 20% más reciente por fecha queda como prueba y no se toca hasta el final.

In [ ]:
print("Train:", train.shape, train[FECHA].min(), "→", train[FECHA].max())
print("Test :", test.shape,  test[FECHA].min(),  "→", test[FECHA].max())
print(f"\n% negativa  train: {train[TARGET].mean()*100:.1f} | test: {test[TARGET].mean()*100:.1f}")
comp = pd.concat([train["banco"].value_counts(normalize=True).rename("% train"),
                  test["banco"].value_counts(normalize=True).rename("% test")], axis=1).mul(100).round(1)
comp["% neg train"] = train.groupby("banco")[TARGET].mean().mul(100).round(1)
comp["% neg test"]  = test.groupby("banco")[TARGET].mean().mul(100).round(1)
display(comp)

**Lectura:** la composición por banco cambia entre train y test (los bancos con ventana corta, como Pichincha, pesan más en el tramo reciente), y la prevalencia de negativas también. Es una limitación real y se reporta; por eso se evaluará además por banco.

## 2. Features derivadas y su evidencia
Solo se usan variables que existen en el momento de publicarse la reseña.

In [ ]:
cand = ["log_n_palabras","n_neg_lexico","es_solo_simbolos","version_app_desconocida"]
filas=[]
for c in cand:
    filas.append({"feature":c,
                  "AUC univariado (train)": round(roc_auc_score(train[TARGET], train[c]),3),
                  "Spearman con target": round(stats.spearmanr(train[c], train[TARGET])[0],3)})
display(pd.DataFrame(filas))
print("\n% negativa según n.º de palabras de queja detectadas:")
display(train.assign(k=train["n_neg_lexico"].clip(upper=3)).groupby("k")[TARGET].agg(["mean","size"]).round(3))

| Feature | Evidencia | ¿Existe al predecir? | Codificación |
|---|---|---|---|
| `log_n_palabras` (hallazgo EDA: % negativa sube de 11% a 80% con la longitud; cola larga) | AUC y Spearman de arriba | Sí: sale del texto | log1p + estandarizado |
| `n_neg_lexico` (palabras de queja) | AUC y tabla por conteo | Sí: sale del texto | conteo + estandarizado |
| `banco` | % negativa por banco 21%–75% | Sí: se conoce la app | One-Hot (`handle_unknown='ignore'`) |
| `comentario_limpio` | TF-IDF 1–2 gramas | Sí | TF-IDF dentro del Pipeline |
| `version_app_desconocida` | 56.5% vs 42.5% negativas | Sí: viene con la reseña | binaria |

## 3. Prueba de que no hay fuga
**(a)** Variables excluidas por no existir al predecir (`me_gusta` se acumula después de publicar; la respuesta del banco ocurre después). Se mide si aportan algo sobre las demás:

In [ ]:
cv = TimeSeriesSplit(5)
def auc_cv(cols):
    m = LogisticRegression(max_iter=1000)
    X = train[cols].astype(float)
    return cross_val_score(m, X, train[TARGET], cv=cv, scoring="roc_auc")
base = ["log_n_palabras","n_neg_lexico","es_solo_simbolos","version_app_desconocida"]
res = {"sin variables prohibidas": auc_cv(base),
       "+ me_gusta": auc_cv(base+["me_gusta"]),
       "+ tiene_respuesta_banco": auc_cv(base+["tiene_respuesta_banco"])}
display(pd.DataFrame({k:[v.mean().round(4), v.std().round(4)] for k,v in res.items()}, index=["AUC media CV","desv."]).T)

**Lectura (a):** `me_gusta` y `tiene_respuesta_banco` no mejoran el AUC una vez que están las demás variables (la diferencia es menor que la desviación entre pliegues). Excluirlas no cuesta nada y evita depender de información que no existe cuando llega una reseña nueva.
**(b)** La preparación está dentro de un `Pipeline` + `ColumnTransformer` (TF-IDF, One-Hot y escalado se ajustan solo con el train de cada pliegue). Las features derivadas son funciones fila a fila.
**(c)** Textos idénticos entre train y test (los comentarios cortos se repiten, p. ej. "excelente app"). Estos casos son fáciles (muy pocos son negativos), así que en la evaluación final se reportará también el resultado sobre los textos no vistos en train:

In [ ]:
tt = set(train[TEXTO]); print(f"% de reseñas de test cuyo texto exacto aparece en train: {test[TEXTO].isin(tt).mean()*100:.1f}")
print(f"Para esas reseñas, % negativa en test: {test.loc[test[TEXTO].isin(tt), TARGET].mean()*100:.1f}")

## 4. Esquema de validación
- **Hold-out:** 20% más reciente (temporal).
- **CV dentro del train:** `TimeSeriesSplit(5)` por fecha. Se usa porque el modelo se aplicará a reseñas futuras y la prevalencia cambia en el tiempo; un `StratifiedKFold` aleatorio mezclaría pasado y futuro y daría un resultado optimista.
- **Métrica técnica:** PR-AUC (`average_precision`) de la clase negativa, con F1 de apoyo.
- **Métrica de negocio:** costo por reseña con costo FN = 5 y FP = 1 (supuesto; sensibilidad 3× y 10×); umbral elegido con datos de entrenamiento.